# Snowdon Structural 2x3 - Initial Setup and IFC Type Summary

This notebook follows the same setup style as the Ifc4 SampleHouse transform notebook.

It does three things first:
- Reads both COBie Excel files from the COBie folder.
- Links to Snowdon+Towers+Sample+Structural2x3.json from JSON Whole Model.
- Prints a summary table of IFC Type counts and lists the IFC Type names found in the JSON.

In [5]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

# ============================================================
# SETUP & PATHS
# ============================================================

workspace_root = Path.cwd().parent
source_json_path = workspace_root / "JSON Whole Model" / "Snowdon+Towers+Sample+Structural2x3.json"
excel_ef_path = workspace_root / "COBie" / "Uniclass2015_EF_v1_16.xlsx"
excel_pr_path = workspace_root / "COBie" / "Uniclass2015_Pr_v1_41.xlsx"

assert source_json_path.exists(), f"JSON not found: {source_json_path}"
assert excel_ef_path.exists(), f"Excel not found: {excel_ef_path}"
assert excel_pr_path.exists(), f"Excel not found: {excel_pr_path}"

print(f"Source JSON: {source_json_path}")
print(f"COBie EF Excel: {excel_ef_path}")
print(f"COBie Pr Excel: {excel_pr_path}")

# ============================================================
# HELPERS
# ============================================================

def normalize_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def normalize_for_match(value):
    return re.sub(r"\s+", " ", normalize_text(value)).strip().casefold()


def read_uniclass_excel(path, preferred_sheet_name):
    attempts = [
        (preferred_sheet_name, 2),
        (0, 2),
        (preferred_sheet_name, 1),
        (0, 1),
        (preferred_sheet_name, 0),
        (0, 0),
    ]

    last_error = None
    for sheet_name, header in attempts:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, header=header)
        except Exception as ex:
            last_error = ex
            continue

        if df.empty:
            continue

        col_names = [normalize_text(c) for c in df.columns]
        has_cobie_col = any(normalize_for_match(c) == "cobie" for c in col_names)
        if has_cobie_col:
            return df

    if last_error is not None:
        raise last_error

    return pd.read_excel(path, sheet_name=0)


def get_properties(item):
    properties = item.get("Properties", []) if isinstance(item, dict) else []
    return properties if isinstance(properties, list) else []


def get_ifc_type(item):
    properties = get_properties(item)
    for prop in properties:
        if not isinstance(prop, dict):
            continue

        category = normalize_text(prop.get("category")).lower()
        display_name = normalize_text(prop.get("displayName")).lower()
        value = normalize_text(prop.get("value"))

        if category == "item" and display_name == "type":
            return value.upper()

    return ""


def choose_lookup_column(df):
    # Prefer 'Title'; otherwise use Excel column G (0-based index 6).
    normalized_cols = {normalize_for_match(col): col for col in df.columns}
    if "title" in normalized_cols:
        return normalized_cols["title"]
    if len(df.columns) > 6:
        return df.columns[6]
    return df.columns[0]


def get_cobie_column(df):
    for col in df.columns:
        if normalize_for_match(col) == "cobie":
            return col
    return None


def first_non_empty_cobie(df, cobie_col):
    for value in df[cobie_col].tolist():
        text = normalize_text(value)
        if text:
            return text
    return ""


# ============================================================
# READ BOTH COBie EXCEL FILES
# ============================================================

ef_df = read_uniclass_excel(excel_ef_path, preferred_sheet_name="EF")
pr_df = read_uniclass_excel(excel_pr_path, preferred_sheet_name="Pr")

print("\nExcel files loaded:")
print(f"- EF rows: {len(ef_df):,}, columns: {len(ef_df.columns)}")
print(f"- Pr rows: {len(pr_df):,}, columns: {len(pr_df.columns)}")

# ============================================================
# LOAD JSON AND BUILD IFC TYPE SUMMARY
# ============================================================

with source_json_path.open("r", encoding="utf-8") as f:
    source_data = json.load(f)

summary_rows = []
for item in source_data:
    ifc_type = get_ifc_type(item)
    if not ifc_type:
        continue
    summary_rows.append({"IFC Type": ifc_type})

summary_df = pd.DataFrame(summary_rows)

if summary_df.empty:
    print("\nNo IFC Type values found in the JSON.")
    ifc_type_count_df = pd.DataFrame(columns=["IFC Type", "Count"])
else:
    ifc_type_count_df = (
        summary_df.groupby("IFC Type", dropna=False)
        .size()
        .rename("Count")
        .reset_index()
        .sort_values(by=["Count", "IFC Type"], ascending=[False, True], kind="stable")
        .reset_index(drop=True)
    )

    print("\nIFC Type summary (count by IFC Type):")
    display(ifc_type_count_df)

    unique_types = ifc_type_count_df["IFC Type"].tolist()
    print(f"\nTotal unique IFC Types: {len(unique_types)}")
    print("IFC Type names:")
    for idx, type_name in enumerate(unique_types, start=1):
        print(f"{idx:>2}. {type_name}")

# ============================================================
# COBie MAPPING (IFC4 SAMPLEHOUSE-STYLE TABLE FORMAT)
# ============================================================

processed_items = [
    {
        "item_type": "Beams",
        "ifc_type": "IFCBEAM",
        "excel_phrase": "Carbon steel beams, columns, channels and tees",
    },
    {
        "item_type": "Slabs",
        "ifc_type": "IFCSLAB",
        "excel_phrase": "Concrete solid slabs",
    },
    {
        "item_type": "Columns",
        "ifc_type": "IFCCOLUMN",
        "excel_phrase": "Carbon steel columns",
    },
    {
        "item_type": "Walls",
        "ifc_type": "IFCWALL",
        "excel_phrase": "Walls",
    },
    {
        "item_type": "Footings",
        "ifc_type": "IFCFOOTING",
        "excel_phrase": "Concrete pocket foundations",
    },
]


def find_cobie_value(phrase):
    phrase_norm = normalize_for_match(phrase)
    sources = [
        ("Uniclass2015_Pr_v1_41.xlsx", pr_df),
        ("Uniclass2015_EF_v1_16.xlsx", ef_df),
    ]

    prepared = []
    for source_name, df in sources:
        lookup_col = choose_lookup_column(df)
        cobie_col = get_cobie_column(df)
        if cobie_col is None:
            continue
        lookup_series = df[lookup_col].astype(str).map(normalize_for_match)
        prepared.append((source_name, df, lookup_col, cobie_col, lookup_series))

    # Pass 1: exact matches in all sources.
    for source_name, df, lookup_col, cobie_col, lookup_series in prepared:
        exact_mask = lookup_series == phrase_norm
        exact_matches = df[exact_mask].copy()
        if not exact_matches.empty:
            cobie_value = first_non_empty_cobie(exact_matches, cobie_col)
            if cobie_value:
                return cobie_value, source_name, lookup_col

    # Pass 2: contains matches in all sources.
    for source_name, df, lookup_col, cobie_col, lookup_series in prepared:
        contains_mask = lookup_series.str.contains(re.escape(phrase_norm), na=False)
        contains_matches = df[contains_mask].copy()
        if not contains_matches.empty:
            cobie_value = first_non_empty_cobie(contains_matches, cobie_col)
            if cobie_value:
                return cobie_value, source_name, lookup_col

    return "", "", ""


ifc_count_map = {}
if not ifc_type_count_df.empty:
    ifc_count_map = dict(zip(ifc_type_count_df["IFC Type"], ifc_type_count_df["Count"]))

mapping_rows = []
for config in processed_items:
    ifc_type = config["ifc_type"]
    phrase = config["excel_phrase"]
    item_type = config["item_type"]

    cobie_value, source_name, lookup_col = find_cobie_value(phrase)
    total_items = int(ifc_count_map.get(ifc_type, 0))

    mapping_rows.append(
        {
            "Item Type": item_type,
            "IFC Type": ifc_type,
            "Total Items": total_items,
            "Name of the model element": phrase,
            "COBie Value": cobie_value,
            "Excel Source": source_name,
            "Lookup Column": str(lookup_col),
        }
    )

mapping_result_df = pd.DataFrame(mapping_rows)

print("\n" + "=" * 80)
print("COBie MAPPING RESULT (REQUESTED IFC TYPES)")
print("=" * 80)
display(mapping_result_df[[
    "Item Type",
    "IFC Type",
    "Total Items",
    "Name of the model element",
    "COBie Value",
]])

print("\nMapping diagnostics (excel source and lookup column):")
display(mapping_result_df[["IFC Type", "Excel Source", "Lookup Column"]])

Source JSON: c:\Git\APS-IFC\JSON Whole Model\Snowdon+Towers+Sample+Structural2x3.json
COBie EF Excel: c:\Git\APS-IFC\COBie\Uniclass2015_EF_v1_16.xlsx
COBie Pr Excel: c:\Git\APS-IFC\COBie\Uniclass2015_Pr_v1_41.xlsx

Excel files loaded:
- EF rows: 231, columns: 14
- Pr rows: 8,451, columns: 14

IFC Type summary (count by IFC Type):


,IFC Type,Count
0,IFCSHAPEREPRESENTATION,4113
1,LCIFCREPRESENTATIONHOLDER,1232
2,IFCMAPPEDITEM,1040
3,IFCBEAM,942
4,IFCPOLYLINE,512
5,IFCEXTRUDEDAREASOLID,486
6,IFCSLAB,108
7,IFCELEMENTASSEMBLY,76
8,IFCCOLUMN,54
9,IFCBUILDINGELEMENTPROXY,50



Total unique IFC Types: 23
IFC Type names:
 1. IFCSHAPEREPRESENTATION
 2. LCIFCREPRESENTATIONHOLDER
 3. IFCMAPPEDITEM
 4. IFCBEAM
 5. IFCPOLYLINE
 6. IFCEXTRUDEDAREASOLID
 7. IFCSLAB
 8. IFCELEMENTASSEMBLY
 9. IFCCOLUMN
10. IFCBUILDINGELEMENTPROXY
11. IFCWALLSTANDARDCASE
12. IFCBOOLEANCLIPPINGRESULT
13. LCOAEXGEOMETRY
14. IFCWALL
15. IFCBUILDINGSTOREY
16. IFCTRIMMEDCURVE
17. COMPOSITE PART
18. IFCGRID
19. IFCFOOTING
20. FILE
21. IFCBUILDING
22. IFCPROJECT
23. IFCSITE

COBie MAPPING RESULT (REQUESTED IFC TYPES)


,Item Type,IFC Type,Total Items,Name of the model element,COBie Value
0,Beams,IFCBEAM,942,"Carbon steel beams, columns, channels and tees","Pr_20_76_51_12 : Carbon steel beams, columns, ..."
1,Slabs,IFCSLAB,108,Concrete solid slabs,Pr_20_85_14_16 : Concrete solid slabs
2,Columns,IFCCOLUMN,54,Carbon steel columns,Pr_20_85_16_11 : Carbon steel columns
3,Walls,IFCWALL,13,Walls,EF_25_10 : Walls
4,Footings,IFCFOOTING,9,Concrete pocket foundations,Pr_20_85_13_65 : Concrete pocket foundations



Mapping diagnostics (excel source and lookup column):


,IFC Type,Excel Source,Lookup Column
0,IFCBEAM,Uniclass2015_Pr_v1_41.xlsx,Title
1,IFCSLAB,Uniclass2015_Pr_v1_41.xlsx,Title
2,IFCCOLUMN,Uniclass2015_Pr_v1_41.xlsx,Title
3,IFCWALL,Uniclass2015_EF_v1_16.xlsx,Title
4,IFCFOOTING,Uniclass2015_Pr_v1_41.xlsx,Title


## Apply COBie Values to JSON_Edit (5 IFC Types)

This step applies the mapped COBie values to `JSON_Edit/Snowdon+Towers+Sample+Structural2x3.json` for only these IFC types:
- IFCBEAM
- IFCSLAB
- IFCCOLUMN
- IFCWALL
- IFCFOOTING

It prints:
- A comparison table (similar format to the Ifc4 SampleHouse workflow)
- A processing summary table
- Output JSON path

In [6]:
from shutil import copy2

# Ensure required inputs from the mapping step exist.
required_vars = ["source_data", "mapping_result_df", "source_json_path", "workspace_root"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(
        "Run Cell 2 first. Missing variables: " + ", ".join(missing_vars)
    )

json_edit_dir = workspace_root / "JSON_Edit"
json_edit_dir.mkdir(parents=True, exist_ok=True)
working_json_path = json_edit_dir / source_json_path.name

if not working_json_path.exists():
    copy2(source_json_path, working_json_path)

with working_json_path.open("r", encoding="utf-8") as f:
    working_data = json.load(f)


def apply_cobie_value(target_item, cobie_value):
    properties = target_item.get("Properties")
    if not isinstance(properties, list):
        properties = []
        target_item["Properties"] = properties

    existing_cobie_prop = None
    for prop in properties:
        if not isinstance(prop, dict):
            continue
        category = normalize_text(prop.get("category"))
        display_name = normalize_text(prop.get("displayName"))
        if category == "IFC" and display_name == "COBie":
            existing_cobie_prop = prop
            break

    before_value = ""
    after_value = normalize_text(cobie_value)

    if existing_cobie_prop is not None:
        before_value = normalize_text(existing_cobie_prop.get("value"))
        if before_value != after_value:
            existing_cobie_prop["value"] = after_value
            return True, before_value, after_value, "Updated"
        return False, before_value, after_value, "Unchanged"

    properties.append(
        {
            "category": "IFC",
            "displayName": "COBie",
            "value": after_value,
        }
    )
    return True, "", after_value, "Added"


def build_comparison_table(processed_rows):
    comparison_df = pd.DataFrame(processed_rows)
    if comparison_df.empty:
        return comparison_df

    comparison_df = comparison_df[[
        "Item Type",
        "IFC Type",
        "Total Items",
        "Name of the model element",
        "COBie Value",
        "Change",
    ]].copy()

    comparison_df = comparison_df.sort_values(
        by=["Item Type", "IFC Type", "Name of the model element"],
        kind="stable",
    ).reset_index(drop=True)

    return comparison_df


# Build IFC type -> mapped COBie value from Cell 2 output.
mapping_lookup = {}
for row in mapping_result_df.to_dict("records"):
    ifc_type = normalize_text(row.get("IFC Type")).upper()
    cobie_value = normalize_text(row.get("COBie Value"))
    item_type = normalize_text(row.get("Item Type"))
    if ifc_type and cobie_value:
        mapping_lookup[ifc_type] = {
            "item_type": item_type,
            "cobie_value": cobie_value,
        }

processed_ifc_types = [
    "IFCBEAM",
    "IFCSLAB",
    "IFCCOLUMN",
    "IFCWALL",
    "IFCFOOTING",
]

processed_rows = []
processing_summary_rows = []

for ifc_type in processed_ifc_types:
    mapped = mapping_lookup.get(ifc_type)
    mapped_cobie = normalize_text((mapped or {}).get("cobie_value"))
    mapped_item_type = normalize_text((mapped or {}).get("item_type")) or ifc_type

    total_items = 0
    updated_count = 0
    added_count = 0
    unchanged_count = 0

    for item in working_data:
        if not isinstance(item, dict):
            continue

        item_ifc_type = get_ifc_type(item)
        if item_ifc_type != ifc_type:
            continue

        total_items += 1

        properties = get_properties(item)
        element_name = normalize_text(item.get("Name"))
        if not element_name:
            for prop in properties:
                if not isinstance(prop, dict):
                    continue
                if normalize_text(prop.get("category")).lower() == "item" and normalize_text(prop.get("displayName")).lower() == "name":
                    element_name = normalize_text(prop.get("value"))
                    break

        if not mapped_cobie:
            change = "No Mapping"
            after_value = ""
        else:
            _, _, after_value, change = apply_cobie_value(item, mapped_cobie)
            if change == "Updated":
                updated_count += 1
            elif change == "Added":
                added_count += 1
            elif change == "Unchanged":
                unchanged_count += 1

        processed_rows.append(
            {
                "Item Type": mapped_item_type,
                "IFC Type": ifc_type,
                "Total Items": total_items,
                "Name of the model element": element_name,
                "COBie Value": after_value if mapped_cobie else "",
                "Change": change,
            }
        )

    processing_summary_rows.append(
        {
            "Item Type": mapped_item_type,
            "IFC Type": ifc_type,
            "Total Items": total_items,
            "Updated": updated_count,
            "Added": added_count,
            "Unchanged": unchanged_count,
            "COBie Value": mapped_cobie,
        }
    )

# Keep Total Items consistent per IFC Type in processed rows.
if processed_rows:
    total_by_ifc = {}
    for row in processed_rows:
        total_by_ifc[row["IFC Type"]] = max(total_by_ifc.get(row["IFC Type"], 0), row["Total Items"])
    for row in processed_rows:
        row["Total Items"] = total_by_ifc.get(row["IFC Type"], row["Total Items"])

with working_json_path.open("w", encoding="utf-8") as f:
    json.dump(working_data, f, ensure_ascii=False, indent=2)

comparison_table_df = build_comparison_table(processed_rows)
processing_summary_df = pd.DataFrame(processing_summary_rows)

print("=" * 80)
print("TABLE COMPARISON")
print("=" * 80)
display(comparison_table_df)

print("\n" + "=" * 80)
print("PROCESSING SUMMARY")
print("=" * 80)
display(processing_summary_df)

print(f"\nOutput Path: {working_json_path}")

TABLE COMPARISON


,Item Type,IFC Type,Total Items,Name of the model element,COBie Value,Change
0,Beams,IFCBEAM,942,C Shapes:C8X11.5:686385,"Pr_20_76_51_12 : Carbon steel beams, columns, ...",Added
1,Beams,IFCBEAM,942,C Shapes:C8X11.5:686634,"Pr_20_76_51_12 : Carbon steel beams, columns, ...",Added
2,Beams,IFCBEAM,942,C Shapes:C8X11.5:687059,"Pr_20_76_51_12 : Carbon steel beams, columns, ...",Added
3,Beams,IFCBEAM,942,C Shapes:C8X11.5:793186,"Pr_20_76_51_12 : Carbon steel beams, columns, ...",Added
4,Beams,IFCBEAM,942,C Shapes:C8X11.5:793189,"Pr_20_76_51_12 : Carbon steel beams, columns, ...",Added
...,...,...,...,...,...,...
1121,Walls,IFCWALL,13,"Basic Wall:Concrete 18"":935926",EF_25_10 : Walls,Added
1122,Walls,IFCWALL,13,"Basic Wall:Concrete 18"":936762",EF_25_10 : Walls,Added
1123,Walls,IFCWALL,13,"Basic Wall:Concrete 18"":936763",EF_25_10 : Walls,Added
1124,Walls,IFCWALL,13,"Basic Wall:Concrete 24"":601472",EF_25_10 : Walls,Added



PROCESSING SUMMARY


,Item Type,IFC Type,Total Items,Updated,Added,Unchanged,COBie Value
0,Beams,IFCBEAM,942,0,942,0,"Pr_20_76_51_12 : Carbon steel beams, columns, ..."
1,Slabs,IFCSLAB,108,0,108,0,Pr_20_85_14_16 : Concrete solid slabs
2,Columns,IFCCOLUMN,54,0,54,0,Pr_20_85_16_11 : Carbon steel columns
3,Walls,IFCWALL,13,0,13,0,EF_25_10 : Walls
4,Footings,IFCFOOTING,9,0,9,0,Pr_20_85_13_65 : Concrete pocket foundations



Output Path: c:\Git\APS-IFC\JSON_Edit\Snowdon+Towers+Sample+Structural2x3.json
